# 📦 Notebook 1: Data Preparation
## Emotion-Conditioned Image Captioning

---

## ✅ What This Notebook Does
1. Mounts Google Drive (your permanent storage across sessions)
2. Downloads the **Flickr8k** dataset (images + captions)
3. Uses **GPT-2** to rewrite neutral captions into 5 emotional styles
4. Builds a vocabulary from the emotional captions
5. Saves everything to Google Drive for use in training

## ⚠️ Before You Run
- Make sure **Runtime → Change runtime type → T4 GPU** is selected
- You need a **Kaggle account** (free) to download Flickr8k — instructions below
- Run cells **one by one** and read the output before proceeding
- This notebook takes ~30–45 minutes total (mostly GPT-2 generation)

---

## Step 1 — Mount Google Drive

All outputs will be saved here so they survive Colab session resets.
Click the link that appears, sign in, copy the code, and paste it below.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/EmotionCaptioning'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
print('✅ Drive mounted. Project folder created at:', PROJECT_DIR)

Mounted at /content/drive
✅ Drive mounted. Project folder created at: /content/drive/MyDrive/EmotionCaptioning


## Step 2 — Install Dependencies

In [3]:
!pip install -q kaggle transformers nltk pycocoevalcap
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('✅ Dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 7.8 MB/s eta 0:00:00
✅ Dependencies installed


## Step 3 — Set Up Kaggle API

### How to get your Kaggle API key:
1. Go to https://www.kaggle.com → click your profile picture → **Account**
2. Scroll to **API** section → click **Create New Token**
3. A file `kaggle.json` will download — open it and copy the values below

Paste your username and key in the cell below:

In [4]:
import os, json

# ✏️ PASTE YOUR KAGGLE CREDENTIALS HERE
KAGGLE_USERNAME = "muhammadasjadali"
KAGGLE_KEY      = "export KAGGLE_API_TOKEN=KGAT_20b281cf26044c2c501b9b796579afb7"

os.makedirs('/root/.config/kaggle', exist_ok=True)
with open('/root/.config/kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('✅ Kaggle credentials saved')

✅ Kaggle credentials saved


## Step 4 — Download Flickr8k

This downloads ~1GB of images and caption files. Takes 3–5 minutes.

> **Note:** You must accept the dataset terms on Kaggle first.
> Visit: https://www.kaggle.com/datasets/adityajn105/flickr8k and click **Download**.

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("adityajn105/flickr8k")

print("Path to dataset files:", path)

100%|██████████| 1.04G/1.04G [00:14<00:00, 78.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/adityajn105/flickr8k/versions/1


In [6]:
import os

DATA_RAW = '/content/flickr8k_raw'
os.makedirs(DATA_RAW, exist_ok=True)

# Download and unzip
!kaggle datasets download -d adityajn105/flickr8k -p {DATA_RAW} --unzip -q
print('✅ Download complete')
!ls {DATA_RAW}

Dataset URL: https://www.kaggle.com/datasets/adityajn105/flickr8k
License(s): CC0-1.0
✅ Download complete
captions.txt  Images


## Step 5 — Parse and Inspect Captions

In [7]:
import os, glob, json, random
import pandas as pd

# Flickr8k caption file is usually captions.txt with format: image#N\tcaption
caption_file = os.path.join(DATA_RAW, 'captions.txt')
if not os.path.exists(caption_file):
    # Try alternate paths
    candidates = glob.glob(DATA_RAW + '/**/*.txt', recursive=True)
    print('Found text files:', candidates)
    caption_file = candidates[0]  # Adjust if needed

df = pd.read_csv(caption_file)
print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
print(df.head(10))

Columns: ['image', 'caption']
Shape: (40455, 2)
                       image  \
0  1000268201_693b08cb0e.jpg   
1  1000268201_693b08cb0e.jpg   
2  1000268201_693b08cb0e.jpg   
3  1000268201_693b08cb0e.jpg   
4  1000268201_693b08cb0e.jpg   
5  1001773457_577c3a7d70.jpg   
6  1001773457_577c3a7d70.jpg   
7  1001773457_577c3a7d70.jpg   
8  1001773457_577c3a7d70.jpg   
9  1001773457_577c3a7d70.jpg   

                                             caption  
0  A child in a pink dress is climbing up a set o...  
1              A girl going into a wooden building .  
2   A little girl climbing into a wooden playhouse .  
3  A little girl climbing the stairs to her playh...  
4  A little girl in a pink dress going into a woo...  
5         A black dog and a spotted dog are fighting  
6  A black dog and a tri-colored dog playing with...  
7  A black dog and a white dog with brown spots a...  
8  Two dogs of different breeds looking at each o...  
9    Two dogs on pavement moving toward each othe

In [8]:
# Standardize column names (handle different Flickr8k formats)
if 'image' in df.columns and 'caption' in df.columns:
    df_caps = df.rename(columns={'image': 'image_id', 'caption': 'caption'})
elif 'image_id' in df.columns:
    df_caps = df
else:
    # Tab-separated format: image#0\tcaption
    df_caps = pd.read_csv(caption_file, sep='\t', names=['image_id', 'caption'])
    df_caps['image_id'] = df_caps['image_id'].apply(lambda x: x.split('#')[0])

# Keep one caption per image for simplicity (first caption)
df_unique = df_caps.drop_duplicates(subset='image_id', keep='first').copy()
df_unique = df_unique[df_unique['caption'].notna()].reset_index(drop=True)

# Verify images exist
img_dir = os.path.join(DATA_RAW, 'Images')
if not os.path.isdir(img_dir):
    img_dir = DATA_RAW  # images may be in root

df_unique['img_path'] = df_unique['image_id'].apply(lambda x: os.path.join(img_dir, x))
df_unique = df_unique[df_unique['img_path'].apply(os.path.exists)].reset_index(drop=True)

print(f'✅ {len(df_unique)} unique images with captions found')
print(df_unique[['image_id','caption']].head(5))

✅ 8091 unique images with captions found
                    image_id  \
0  1000268201_693b08cb0e.jpg   
1  1001773457_577c3a7d70.jpg   
2  1002674143_1b742ab4b8.jpg   
3  1003163366_44323f5815.jpg   
4  1007129816_e794419615.jpg   

                                             caption  
0  A child in a pink dress is climbing up a set o...  
1         A black dog and a spotted dog are fighting  
2  A little girl covered in paint sits in front o...  
3  A man lays on a bench while his dog sits by him .  
4     A man in an orange hat starring at something .  


## Step 6 — Generate Emotion Captions with GPT-2

This is the **core data augmentation step**. We take neutral Flickr8k captions and rewrite them into 5 emotional styles using GPT-2.

The 5 emotions are: `joyful`, `sad`, `tense`, `romantic`, `humorous`

**This takes ~20–35 minutes on free Colab T4.** Get a coffee ☕

> We use **sampling-based generation** (temperature=0.9) for diverse outputs, not greedy decoding.

In [10]:
# 1. Install/Update the necessary libraries
!pip install -U bitsandbytes accelerate transformers

# 2. IMPORTANT: After this finishes, go to the top menu:
# Runtime -> Restart Session (or Restart Runtime)
# This clears the internal Python state so it can load the bitsandbytes C libraries.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 90.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [1]:


# from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# model_id = "mistralai/Mistral-7B-Instruct-v0.3" # You can also use "mistralai/Mistral-7B-Instruct-v0.3"

# tokenizer = AutoTokenizer.from_pretrained(model_id)
# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     device_map="auto",

# )

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# 1. Configure 4-bit quantization (The "Magic" part)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Critical for batching/padding

# 3. Load Model with the config
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto", # This will now put the WHOLE model on GPU
)

print("✅ Model loaded in 4-bit. No more offloading!")

def generate_emotion_caption(neutral_caption, emotion):
    # Llama-3 works best with a clear system prompt
    messages = [
        {"role": "system", "content": f"You are a creative writer. Rewrite the user's image caption to sound {emotion}. Keep it to one short sentence and do not add new objects. No intro text."},
        {"role": "user", "content": neutral_caption},
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    return response.strip()

KeyboardInterrupt: 

In [9]:
test_caption = "A child in a pink dress is climbing up a set of stairs in an entry way."
emotions_to_test = ['joyful', 'sad', 'tense', 'romantic', 'humorous']

print(f"Original: {test_caption}\n" + "-"*50)

for emo in emotions_to_test:
    result = generate_emotion_caption(test_caption, emo)
    print(f"{emo.capitalize():10} : {result}")

Original: A child in a pink dress is climbing up a set of stairs in an entry way.
--------------------------------------------------
Joyful     : Little Pink Princess, Climbing up Stairway to Castle Adventures!
Sad        : A little girl in a pink dress ascends the steps, each footfall echoing a silent goodbye.
Tense      : A spirited child ascends the entryway's staircase, her pink dress billowing with each determined step.
Romantic   : A timeless vision, a little darling gracefully ascends the grand staircase, each step a testament to an enchanting journey.
Humorous   : A tiny pink superhero, undeterred by gravity, scales the stairway to heaven with a snack-sized cape flapping behind.


In [12]:
output_path = '/content/drive/MyDrive/EmotionCaptioning/flickr8k_emotional_captions.csv'
# Define the emotion categories and their corresponding IDs
EMOTIONS = ['joyful', 'sad', 'tense', 'romantic', 'humorous']
EMOTION_TO_ID = {e: i for i, e in enumerate(EMOTIONS)}

In [1]:
from tqdm.auto import tqdm

start_idx = 0
if os.path.exists(output_path):
    try:
        existing_df = pd.read_csv(output_path)
        # Find how many unique images we've already done
        done_images = existing_df['image_id'].unique()
        start_idx = len(done_images)
        print(f"⏩ Resuming from index {start_idx}. Already processed {len(done_images)} images.")
    except Exception as e:
        print(f"Starting fresh: {e}")

# 3. Stream to File
# We use 'a' (append) mode if resuming, 'w' (write) if starting fresh
for i in tqdm(range(start_idx, len(df_unique)), desc='Generating captions'):
    row = df_unique.iloc[i]
    image_records = []

    # Generate all 5 emotions for this specific image
    for emotion in EMOTIONS:
        try:
            emo_cap = generate_emotion_caption(row['caption'], emotion)
            image_records.append({
                'image_id': row['image_id'],
                'img_path': row['img_path'],
                'neutral_caption': row['caption'],
                'emotion': emotion,
                'emotion_id': EMOTION_TO_ID[emotion],
                'emotion_caption': emo_cap,
            })
        except Exception as e:
            print(f"Error at image {row['image_id']} for emotion {emotion}: {e}")
            continue

    # Convert just these 5 rows to a small DF and append to CSV
    batch_df = pd.DataFrame(image_records)

    # Write header only if file doesn't exist (first run)
    write_header = not os.path.exists(output_path)
    batch_df.to_csv(output_path, mode='a', index=False, header=write_header)

    # Explicitly clear the small list to save RAM
    del image_records

NameError: name 'os' is not defined

## Step 7 — Build Vocabulary

In [ ]:
from collections import Counter
import re

def tokenize(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s']", '', text)
    return text.split()

# Count word frequencies
counter = Counter()
for cap in df_emotion['emotion_caption']:
    counter.update(tokenize(cap))

MIN_FREQ = 2  # Words appearing less than this are replaced with <UNK>

vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
for word, freq in counter.items():
    if freq >= MIN_FREQ:
        vocab[word] = len(vocab)

idx_to_word = {v: k for k, v in vocab.items()}

print(f'✅ Vocabulary size: {len(vocab)} words')
print('Sample vocab:', list(vocab.items())[:20])

## Step 8 — Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Split by image_id to avoid data leakage
all_image_ids = df_emotion['image_id'].unique()
train_ids, temp_ids = train_test_split(all_image_ids, test_size=0.2, random_state=42)
val_ids, test_ids   = train_test_split(temp_ids, test_size=0.5, random_state=42)

df_train = df_emotion[df_emotion['image_id'].isin(train_ids)].reset_index(drop=True)
df_val   = df_emotion[df_emotion['image_id'].isin(val_ids)].reset_index(drop=True)
df_test  = df_emotion[df_emotion['image_id'].isin(test_ids)].reset_index(drop=True)

print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')

## Step 9 — Save Everything to Google Drive

In [ ]:
import pickle

df_train.to_csv(f'{PROJECT_DIR}/data/train.csv', index=False)
df_val.to_csv(f'{PROJECT_DIR}/data/val.csv', index=False)
df_test.to_csv(f'{PROJECT_DIR}/data/test.csv', index=False)

with open(f'{PROJECT_DIR}/data/vocab.pkl', 'wb') as f:
    pickle.dump({'vocab': vocab, 'idx_to_word': idx_to_word, 'emotions': EMOTIONS}, f)

# Save config for other notebooks
config = {
    'PROJECT_DIR': PROJECT_DIR,
    'IMG_DIR': img_dir,
    'VOCAB_SIZE': len(vocab),
    'EMOTIONS': EMOTIONS,
    'EMOTION_TO_ID': EMOTION_TO_ID,
    'N_EMOTIONS': len(EMOTIONS),
    'EMBED_DIM': 64,
    'HIDDEN_DIM': 512,
    'VISUAL_DIM': 2048,   # ResNet-50
    'FUSED_DIM': 512,
    'MAX_SEQ_LEN': 40,
}
with open(f'{PROJECT_DIR}/data/config.pkl', 'wb') as f:
    pickle.dump(config, f)

print('✅ All data saved to Google Drive!')
print(f'  Train: {len(df_train)} samples')
print(f'  Val:   {len(df_val)} samples')
print(f'  Test:  {len(df_test)} samples')
print(f'  Vocab: {len(vocab)} words')

## Step 10 — Quick Data Sanity Check

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Show a few examples
sample = df_train.sample(5, random_state=0)
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (_, row) in zip(axes, sample.iterrows()):
    img = Image.open(row['img_path']).convert('RGB')
    ax.imshow(img)
    ax.set_title(f"{row['emotion']}\n{row['emotion_caption'][:60]}...", fontsize=7, wrap=True)
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/data/sample_viz.png', dpi=100)
plt.show()
print('✅ Sample visualization saved')

In [ ]:
# Emotion distribution check
df_train['emotion'].value_counts().plot(kind='bar', title='Emotion distribution (train)')
plt.tight_layout()
plt.show()
print('✅ Distribution looks balanced (each emotion ~20%)')

---
## ✅ Notebook 1 Complete!

### What was saved to Google Drive:
| File | Contents |
|------|----------|
| `data/train.csv` | Training triples (image_id, emotion, caption) |
| `data/val.csv`   | Validation triples |
| `data/test.csv`  | Test triples |
| `data/vocab.pkl` | Word → index mapping |
| `data/config.pkl`| Model hyperparameter config |

### ➡️ Next Step
Open **Notebook 2: Model Training** and run it.
The images stay in `/content/flickr8k_raw` — if your session resets, re-run the download cell.